# 1: Reshaping produces different results in eval()

minimal, runnable PyTorch example that provably produces different outputs in eval() when you train BN with one reshape and evaluate with another.

🔴 Key idea demonstrated (picture)

![alt text](readme_imgs/batch_norm/bn_nuance1.png)

🔴 Key idea demonstrated
- Training: BN sees data as [B, E, S]
- Evaluation: BN sees data as [B*S, E]
- Same BN module
- Same logical data
- Different outputs in eval()

🔴 Why this happens 

![alt text](readme_imgs/batch_norm/bn_nuance1.png)

🔴 Key Takeaway :
- Retain the same shape of X in both training and eval . Either [B,E,S] or [B*S, E] either one.

🔴 KSW NOTE
- in [B,E,S] and [B*S, E]. The mathematical updates are the same. Even the running mean and variance updates are mathematically the same
- the number of batch updates can remain the same as well
- hence not sure why Eval1 and Eval2 are different.
- Research when you have excess time at disposal . LOL 😂

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# ----- toy dimensions -----
B, S, E = 2, 3, 4

# ----- fixed input -----
x = torch.tensor(
    [
        [[ 1.0,  2.0,  3.0,   4.0],
         [ 2.0,  4.0,  6.0,   8.0],
         [20.0,  9.0,  2.0,   0.1]],

        [[ 0.0,  5.0, 10.0, 150.0],
         [ 1.0,  1.0,  1.0,   1.0],
         [ 2.0,  3.0,  4.0,   5.0]]
    ]
)  # shape [B,S,E]

bn = nn.BatchNorm1d(E, momentum=0.1)

# =========================
# TRAINING PHASE
# =========================
bn.train()

for _ in range(50):
    # training uses [B,E,S]
    x_train = x.transpose(1, 2)   # [B,S,E] → [B,E,S]
    _ = bn(x_train)

# Save running stats
running_mean = bn.running_mean.clone()
running_var  = bn.running_var.clone()

# =========================
# EVALUATION PHASE
# =========================
bn.eval()

# Case A: evaluate SAME way as training
x_eval_A = x.transpose(1, 2)       # [B,E,S]
y_A = bn(x_eval_A).transpose(1, 2) # back to [B,S,E]

# Case B: evaluate with FLATTENED tokens
x_eval_B = x.reshape(B * S, E)     # [B*S,E]
y_B = bn(x_eval_B).reshape(B, S, E) # back to [B,S,E]

# =========================
# Compare
# =========================
print("Running mean:", running_mean)
print("Running var :", running_var)
print()

print("Eval output (trained-shape):")
print(y_A)
print()

print("Eval output (flattened-shape):")
print(y_B)
print()

print("Max absolute difference:",
      (y_A - y_B).abs().max().item())

print("Are they exactly equal?", torch.allclose(y_A, y_B, atol=1e-15))


Running mean: tensor([ 4.3110,  3.9794,  4.3110, 27.8723])
Running var : tensor([  59.1653,    7.9639,   10.6168, 3560.8401])

Eval output (trained-shape):
tensor([[[-0.4305, -0.7014, -0.4024, -0.4001],
         [-0.3004,  0.0073,  0.5184, -0.3330],
         [ 2.0397,  1.7791, -0.7093, -0.4654]],

        [[-0.5605,  0.3617,  1.7460,  2.0466],
         [-0.4305, -1.0558, -1.0162, -0.4503],
         [-0.3004, -0.3470, -0.0954, -0.3833]]], grad_fn=<TransposeBackward0>)

Eval output (flattened-shape):
tensor([[[-0.4305, -0.7014, -0.4024, -0.4001],
         [-0.3004,  0.0073,  0.5184, -0.3330],
         [ 2.0397,  1.7791, -0.7093, -0.4654]],

        [[-0.5605,  0.3617,  1.7460,  2.0466],
         [-0.4305, -1.0558, -1.0162, -0.4503],
         [-0.3004, -0.3470, -0.0954, -0.3833]]], grad_fn=<ViewBackward0>)

Max absolute difference: 1.1920928955078125e-07
Are they exactly equal? True


🔴 KSW NOTE
- in [B,E,S] and [B*S, E]. The mathematical updates are the same. Even the running mean and variance updates are mathematically the same
- the number of batch updates can remain the same as well
- hence not sure why Eval1 and Eval2 are different.
- Research when you have excess time at disposal . LOL 😂

In [2]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# ----- toy dimensions -----
B, S, E = 2, 3, 4

# ----- fixed input in double precision -----
x = torch.tensor(
    [
        [[ 1.0,  2.0,  3.0,   4.0],
         [ 2.0,  4.0,  6.0,   8.0],
         [20.0,  9.0,  2.0,   0.1]],

        [[ 0.0,  5.0, 10.0, 150.0],
         [ 1.0,  1.0,  1.0,   1.0],
         [ 2.0,  3.0,  4.0,   5.0]]
    ], dtype=torch.float64  # double precision
)

bn = nn.BatchNorm1d(E, momentum=0.1, dtype=torch.float64)

# =========================
# TRAINING PHASE
# =========================
bn.train()
for _ in range(50):
    x_train = x.transpose(1, 2)   # [B,S,E] → [B,E,S]
    _ = bn(x_train)

bn.eval()

# Case A: same shape
y_A = bn(x.transpose(1, 2)).transpose(1, 2)

# Case B: flattened tokens
y_B = bn(x.reshape(B*S, E)).reshape(B, S, E)

# Compare
print("Max absolute difference:", (y_A - y_B).abs().max().item())
print("Are they exactly equal?", torch.allclose(y_A, y_B, atol=1e-15))


Max absolute difference: 4.440892098500626e-16
Are they exactly equal? True
